In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data/retail.csv')
df

,Quantity,StockCode,CustomerID
0,6,85123A,17850.0
1,6,71053,17850.0
2,8,84406B,17850.0
3,6,84029G,17850.0
4,6,84029E,17850.0
...,...,...,...
397919,12,22613,12680.0
397920,6,22899,12680.0
397921,4,23254,12680.0
397922,4,23255,12680.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 397924 entries, 0 to 397923
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Quantity    397924 non-null  int64  
 1   StockCode   397924 non-null  str    
 2   CustomerID  397924 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 11.0 MB


### Part (a)

In [5]:
sc_unique = df['StockCode'].nunique()
ci_unique = df['CustomerID'].nunique()
print(f"Nb of unique Stock Codes : {sc_unique}, Nb of unique Customer ID's : {ci_unique}")

Nb of unique Stock Codes : 3665, Nb of unique Customer ID's : 4339


We see that there are 3665 distinct stock code categories, and 4339 distinct customer ID categories. Here, one-hot encoding will result in a huge number of features (3665 for stock, 4339 for customer id). This is highly 
likely to degrade performance.

### Part (b)

In [6]:
import category_encoders as ce
from sklearn.model_selection import train_test_split

In [7]:
#We first create the train/val/test sets, and then compute target encoding.
X = df[['StockCode','CustomerID']]
y = df['Quantity']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42
)

te = ce.TargetEncoder(cols=['StockCode','CustomerID']) 
te.fit(X_train, y_train) 
X_train_enc = te.transform(X_train)
X_val_enc = te.transform(X_val)
X_test_enc  = te.transform(X_test)

In [9]:
X_train_enc.info()

<class 'pandas.DataFrame'>
Index: 238754 entries, 110491 to 101825
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   StockCode   238754 non-null  float64
 1   CustomerID  238754 non-null  float64
dtypes: float64(2)
memory usage: 5.5 MB


I thought fitting a linear regression using MAE as our loss function would be appropiate, since it is less sensitive to outliers. \
However, since there are 397924 data points, using the method in sklearn took so long to compute. That's why I used SGD regression instead. \
(Same but different optimization)

In [21]:
y_test.describe()

count    79585.000000
mean        13.481146
std        266.863626
min          1.000000
25%          2.000000
50%          5.000000
75%         12.000000
max      74215.000000
Name: Quantity, dtype: float64

In [22]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

model = SGDRegressor(loss="epsilon_insensitive", epsilon=0,random_state=42)

model.fit(X_train_enc, y_train)

y_test_pred = model.predict(X_test_enc)
print(f"Test MAE: {mean_absolute_error(y_test, y_test_pred):.2f}")

Test MAE: 8.67


### Part (c)

Now, to learn the embeddings using an MLP, we have use the nn.Embedding lookup table in PyTorch, this method takes IDs (0,1,2,...) so we first have to convert the categorical variables to integer IDs, and produce the weights, and then \
our goal is to feed this weights to the MLP, to learn the weights correlated with the output labels. e.g. a two customers who have a closed quantity of stocks will \
have similar weights. In addition, since we feed the two embeddings of two distinct features, the model also learns the relations between these features, which target encoding is not capable of doing.

In [26]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [32]:
#We build the mappings from the training set only
stock_map = {c: i + 1 for i, c in enumerate(X_train["StockCode"].unique())}
cust_map  = {c: i + 1 for i, c in enumerate(X_train["CustomerID"].unique())}

#+1 for the unknown category (ID 0)
n_stock = len(stock_map) + 1
n_cust  = len(cust_map) + 1
print(n_stock, n_cust)

3565 4294


In [33]:
#We store our data in PyTorch tensors.
def to_ids(X, col, mapping):
    #Unseen categories are mapped to 0
    return torch.tensor(X[col].map(mapping).fillna(0).astype(int).to_numpy(), dtype=torch.long)

X_train_stock = to_ids(X_train, "StockCode", stock_map)
X_train_cust  = to_ids(X_train, "CustomerID", cust_map)
y_train_t     = torch.tensor(y_train.to_numpy(), dtype=torch.float32)

X_val_stock = to_ids(X_val, "StockCode", stock_map)
X_val_cust  = to_ids(X_val, "CustomerID", cust_map)

X_test_stock = to_ids(X_test, "StockCode", stock_map)
X_test_cust  = to_ids(X_test, "CustomerID", cust_map)

print(X_train_stock.shape, X_train_cust.shape, y_train_t.shape)

torch.Size([238754]) torch.Size([238754]) torch.Size([238754])


In [34]:
#We define our hypothesis class. Each categorical feature has its own embedding layer.
class EmbeddingMLP(nn.Module):
    def __init__(self, n_stock, n_cust, stock_dim, cust_dim, hidden=64):
        super().__init__()
        #We need 2 embeddings for our 2 categorical features
        self.stock_emb = nn.Embedding(n_stock, stock_dim)
        self.cust_emb  = nn.Embedding(n_cust, cust_dim)
        #The MLP takes the concatenated embeddings as input
        self.mlp = nn.Sequential(
            nn.Linear(stock_dim + cust_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, stock_ids, cust_ids):
        x = torch.cat([self.stock_emb(stock_ids), self.cust_emb(cust_ids)], dim=1)
        #We squeeze so the output has shape (batch,), like y
        return self.mlp(x).squeeze(1)

In [35]:
#Folk wisdom
stock_dim = int(np.sqrt(n_stock))
cust_dim  = int(np.sqrt(n_cust))
print(stock_dim, cust_dim)

torch.manual_seed(42)
model = EmbeddingMLP(n_stock, n_cust, stock_dim, cust_dim)

59 65


In [37]:
#We define a training algorithm. It is the same as before, except the model takes two inputs.
def train_emb_mlp(model, optimizer, criterion, X_stock, X_cust, y, batch_size, n_epochs):
    #We calculate the nb of rows to sample random batches
    nb_rows = X_stock.shape[0]
    for epoch in range(n_epochs):
        model.train()
        #We generate random indices
        indices = torch.randperm(nb_rows)
        for start in range(0, nb_rows, batch_size):
            batch_indices = indices[start:start + batch_size]
            y_pred = model(X_stock[batch_indices], X_cust[batch_indices])
            optimizer.zero_grad()
            loss = criterion(y_pred, y[batch_indices])
            loss.backward()
            optimizer.step()
        print(f"Epoch: {epoch}, Loss: {loss.item()}")

L1 Loss to train

In [38]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.L1Loss()

train_emb_mlp(model, optimizer, criterion, X_train_stock, X_train_cust, y_train_t, 256, 10)

Epoch: 0, Loss: 5.460529804229736
Epoch: 1, Loss: 6.3958210945129395
Epoch: 2, Loss: 6.759342193603516
Epoch: 3, Loss: 4.881342887878418
Epoch: 4, Loss: 3.4895031452178955
Epoch: 5, Loss: 3.907503128051758
Epoch: 6, Loss: 9.800168991088867
Epoch: 7, Loss: 3.5543465614318848
Epoch: 8, Loss: 3.372739791870117
Epoch: 9, Loss: 7.053396701812744


In [39]:
#We validate the model
model.eval()
with torch.no_grad():
    y_val_pred = model(X_val_stock, X_val_cust)

emb_mae_val = mean_absolute_error(y_val, y_val_pred.numpy())
print(f"Embedding MLP validation MAE: {emb_mae_val:.2f}")

Embedding MLP validation MAE: 7.04


Now we trained and validated the model, we can compare it with the target encoding in part b

In [41]:
with torch.no_grad():
    y_test_pred_emb = model(X_test_stock, X_test_cust)
emb_mae_test = mean_absolute_error(y_test, y_test_pred_emb.numpy())

#Target encoding model from part (b)
te_mae_test = mean_absolute_error(y_test, y_test_pred)

#Median baseline
median_mae_test = mean_absolute_error(y_test, np.full(len(y_test), y_train.median()))

pd.DataFrame({
    "Model": ["Median baseline", "Target encoding + linear", "Embedding MLP"],
    "Test MAE": [median_mae_test, te_mae_test, emb_mae_test]
}).round(3)

,Model,Test MAE
0,Median baseline,11.277
1,Target encoding + linear,8.666
2,Embedding MLP,6.891


We see that the embedding MLP gives a test MAE of 6.89, which is better than the target encoding model (8.67) and the median baseline (11.28).

We think this is because of two reasons. First, target encoding represents each category with a single number (the mean quantity of that stock code or customer), \
while the embedding layer learns a vector for each category. Since these vectors are learned together with the MLP, they become correlated with the output labels, \
e.g. two customers who buy similar quantities end up with similar embeddings. So the model can store more information about each category than a single mean.

Second, since we feed the two embeddings together to the MLP, the model can also learn the relations between the two features. \
Target encoding with a linear model is not capable of capturing this.
The MLP can learn these interactions thanks to its non-linear (ReLU) layers.

Finally, the validation MAE (7.04) and the test MAE (6.89) are close, so the model is not overfitting.

### Part (d)

In [42]:
from sklearn.preprocessing import StandardScaler

#We reuse the target encoded features from part (b) and standardize them
scaler = StandardScaler()
X_train_num = torch.tensor(scaler.fit_transform(X_train_enc), dtype=torch.float32)
X_val_num   = torch.tensor(scaler.transform(X_val_enc), dtype=torch.float32)
X_test_num  = torch.tensor(scaler.transform(X_test_enc), dtype=torch.float32)
print(X_train_num.shape)

torch.Size([238754, 2])


In [43]:
#We define our hypothesis class. The input is the two embeddings + the two target encoded features.
class CombinedMLP(nn.Module):
    def __init__(self, n_stock, n_cust, stock_dim, cust_dim, n_numeric, hidden=64):
        super().__init__()
        self.stock_emb = nn.Embedding(n_stock, stock_dim)
        self.cust_emb  = nn.Embedding(n_cust, cust_dim)
        self.mlp = nn.Sequential(
            nn.Linear(stock_dim + cust_dim + n_numeric, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, stock_ids, cust_ids, x_num):
        x = torch.cat([self.stock_emb(stock_ids), self.cust_emb(cust_ids), x_num], dim=1)
        return self.mlp(x).squeeze(1)

In [44]:
#Same training algorithm, with three inputs
def train_combined(model, optimizer, criterion, X_stock, X_cust, X_num, y, batch_size, n_epochs):
    nb_rows = X_stock.shape[0]
    for epoch in range(n_epochs):
        model.train()
        indices = torch.randperm(nb_rows)
        for start in range(0, nb_rows, batch_size):
            batch_indices = indices[start:start + batch_size]
            y_pred = model(X_stock[batch_indices], X_cust[batch_indices], X_num[batch_indices])
            optimizer.zero_grad()
            loss = criterion(y_pred, y[batch_indices])
            loss.backward()
            optimizer.step()
        print(f"Epoch: {epoch}, Loss: {loss.item()}")

In [45]:
torch.manual_seed(42)
model_comb = CombinedMLP(n_stock, n_cust, stock_dim, cust_dim, n_numeric=2)
optimizer = torch.optim.Adam(model_comb.parameters(), lr=0.001)
criterion = nn.L1Loss()

train_combined(model_comb, optimizer, criterion,
               X_train_stock, X_train_cust, X_train_num, y_train_t, 256, 10)

Epoch: 0, Loss: 8.388290405273438
Epoch: 1, Loss: 10.20163345336914
Epoch: 2, Loss: 6.097353935241699
Epoch: 3, Loss: 7.156831741333008
Epoch: 4, Loss: 7.06122350692749
Epoch: 5, Loss: 3.7183427810668945
Epoch: 6, Loss: 3.9752767086029053
Epoch: 7, Loss: 4.721875190734863
Epoch: 8, Loss: 5.955386161804199
Epoch: 9, Loss: 13.206613540649414


In [46]:
#We validate the model
model_comb.eval()
with torch.no_grad():
    y_val_pred_comb = model_comb(X_val_stock, X_val_cust, X_val_num)
print(f"Combined model validation MAE: {mean_absolute_error(y_val, y_val_pred_comb.numpy()):.2f}")

Combined model validation MAE: 7.06


In [47]:
#We compare the models on the test set
with torch.no_grad():
    y_test_pred_comb = model_comb(X_test_stock, X_test_cust, X_test_num)
comb_mae_test = mean_absolute_error(y_test, y_test_pred_comb.numpy())

pd.DataFrame({
    "Model": ["Median baseline", "Target encoding + linear", "Embedding MLP", "Embedding + target encoding MLP"],
    "Test MAE": [median_mae_test, te_mae_test, emb_mae_test, comb_mae_test]
}).round(3)

,Model,Test MAE
0,Median baseline,11.277
1,Target encoding + linear,8.666
2,Embedding MLP,6.891
3,Embedding + target encoding MLP,6.924


### Part (e)

For the embedding layer, each stock code and each customer has its own vector, which is initialized randomly (using pytoch nn.Embedding) \
and then learned together with the MLP by gradient descent. Since the vectors are updated in training \
they become correlated with the output labels: two customers who buy similar quantities, or two products that are bought in similar quantities, \
end up with similar embeddings. The embedding layer learns a representation of each category for predicting the quantity.

Since there are thousands of stock codes and customers, so one-hot encoding would create \
a huge number of sparse features, but embeddings represent each category with a small vector. \
Compared to target encoding, which summarizes each category with a single mean, an embedding vector can store more information \
about each category. And since the two embeddings are fed together to the MLP, the model can also learn the relations between customers and products, \
e.g. a customer who buys large quantities of some products but not others. The target encoding model with a linear regression can only \
add the two effects separately, so it cannot capture this.

The results on the test set agree with this intuition: the embedding MLP (test MAE 6.89) clearly outperforms the target encoding model (8.67), \
and both improve over the median baseline (11.28).